# Telco customer churn — preprocessing pipeline

Cleans the raw IBM Telco dataset and produces train/test splits plus a fitted `ColumnTransformer` saved as a joblib artifact. Steps:

1. Drop `customerID`; encode `Churn` → 0/1
2. Impute 11 missing `TotalCharges` rows (median of non-missing rows)
3. 80/20 stratified train/test split (`random_state=42`)
4. `ColumnTransformer`: `StandardScaler` on numerics, `OneHotEncoder` on categoricals
5. `fit_transform` on train, `transform` on test only
6. Save `models/preprocessor.joblib`, `data/processed/train.csv`, `data/processed/test.csv`

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path("..").resolve()
RAW = ROOT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
PROCESSED = ROOT / "data/processed"
MODELS = ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

## Load and initial clean

In [2]:
df = pd.read_csv(RAW)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df.shape

(7043, 20)

## Impute missing TotalCharges

11 rows have `TotalCharges = NaN` (blank strings in the CSV). These are new customers with `tenure = 0` and therefore zero or undefined total charges. Imputing with the **training-set median** avoids data leakage and is a conservative choice; the 11 rows are retained rather than dropped so the full cohort is modelled.

In [3]:
print(f"Missing TotalCharges before imputation: {df['TotalCharges'].isna().sum()}")
tc_median = df["TotalCharges"].median()
df["TotalCharges"] = df["TotalCharges"].fillna(tc_median)
print(f"Missing TotalCharges after imputation:  {df['TotalCharges'].isna().sum()}")

Missing TotalCharges before imputation: 11
Missing TotalCharges after imputation:  0


## Train / test split (80/20, stratified)

In [4]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.4f}  test: {y_test.mean():.4f}")

Train: (5634, 19)  Test: (1409, 19)
Churn rate — train: 0.2654  test: 0.2654


## ColumnTransformer: scale numerics, one-hot encode categoricals

In [5]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numeric ({len(numeric_cols)}):     {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
print(f"\nTransformed train shape: {X_train_transformed.shape}")
print(f"Transformed test shape:  {X_test_transformed.shape}")

Numeric (4):     ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Transformed train shape: (5634, 45)
Transformed test shape:  (1409, 45)


C:\Users\Admin\AppData\Local\Temp\ipykernel_452\56647468.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


## Save artifacts

In [6]:
feature_names = (
    numeric_cols
    + preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols).tolist()
)

train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
train_df.insert(0, "Churn", y_train.values)

test_df = pd.DataFrame(X_test_transformed, columns=feature_names)
test_df.insert(0, "Churn", y_test.values)

train_df.to_csv(PROCESSED / "train.csv", index=False)
test_df.to_csv(PROCESSED / "test.csv", index=False)
joblib.dump(preprocessor, MODELS / "preprocessor.joblib")

print(f"Saved: {PROCESSED / 'train.csv'}")
print(f"Saved: {PROCESSED / 'test.csv'}")
print(f"Saved: {MODELS / 'preprocessor.joblib'}")

Saved: C:\Users\Admin\Documents\Postgraduate\MDSAI\CS5998 - Capstone Project\churn-prediction\data\processed\train.csv
Saved: C:\Users\Admin\Documents\Postgraduate\MDSAI\CS5998 - Capstone Project\churn-prediction\data\processed\test.csv
Saved: C:\Users\Admin\Documents\Postgraduate\MDSAI\CS5998 - Capstone Project\churn-prediction\models\preprocessor.joblib


## Quick sanity check

In [7]:
train_loaded = pd.read_csv(PROCESSED / "train.csv")
test_loaded = pd.read_csv(PROCESSED / "test.csv")
print(f"train.csv: {train_loaded.shape}  churn rate: {train_loaded['Churn'].mean():.4f}")
print(f"test.csv:  {test_loaded.shape}  churn rate: {test_loaded['Churn'].mean():.4f}")
train_loaded.head(3)

train.csv: (5634, 46)  churn rate: 0.2654
test.csv:  (1409, 46)  churn rate: 0.2654


,Churn,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,-0.441773,0.102371,-0.521976,-0.263289,0.0,1.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0,-0.441773,-0.711743,0.337478,-0.504814,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0,-0.441773,-0.793155,-0.809013,-0.751213,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
